# Order-book analysis and model lab

Offline analysis of explicitly stopped, integrity-checked exports. **Default: SYNTHETIC integration data, not empirical evidence.** No broker, token, database, new subscription or order is accessed. This notebook extends the established date-split liquidity engine; see `docs/orderbook-model-lab.md` for source mapping, quantity/clock conventions and exclusions.

For private data, copy this notebook outside Git and set `DTS_REPO_ROOT` and `DTS_LIQUIDITY_MANIFEST` before starting a fresh kernel. Keep executed private outputs out of version control.

In [ ]:
import os, sys
from pathlib import Path
root = Path(os.environ['DTS_REPO_ROOT']).expanduser().resolve() if os.environ.get('DTS_REPO_ROOT') else next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'tools/orderbook_lab.py').is_file())
sys.path[:0] = [str(root/'tools'), str(root/'research/liquidity_aware_hedging')]
from orderbook_lab import analyze_many, write_report
from orderbook.features import Config
from liquidity_study import prepare_inputs
from orderbook.hedge import synthetic_case
import pandas as pd
manifest = os.environ.get('DTS_LIQUIDITY_MANIFEST')
data, learning_config, split = prepare_inputs(manifest=manifest) if manifest else prepare_inputs(synthetic=True)
print('OBSERVED exports; no performance has yet been established.' if manifest else 'SYNTHETIC DEMONSTRATION — not market evidence')

## 1. Measurement and coverage
Inspect observed rows, distinct price levels, side-age/reset exclusions, incomplete five-minute variance diagnostics and targets beyond displayed capacity. Features come from archived callbacks, not the polled terminal display. Size-weighted best price and book-crossing cost are proxies, not fills.

In [ ]:
q = learning_config.quantity * (-1 if learning_config.target == 'sell_cost_bps' else 1)
config = Config(quantity=q, step_seconds=learning_config.step_seconds, horizon_seconds=learning_config.horizon_seconds, lookback_seconds=learning_config.lookback_seconds, stale_seconds=learning_config.max_side_age_seconds, levels=5)
result = analyze_many(data, config, learning_config, split)
display(pd.DataFrame([dict(session=s['identity']['session_id'], venue=s['identity']['venue'], events=s['event_count'], model_rows=len(s['samples']), grid_quality=s['quality_counts']) for s in result['sessions']]))

## 2. Chronological prediction
The primary label is future displayed crossing cost (default 30 seconds), in basis points. Persistence and residual ridge are retained. The new nonlinear model adds the average residual of 25 nearest **training** states to current liquidity. All feature families share eligible rows. UTC dates stay together; preprocessing is train-only; validation selects; test evaluates. `history` contains current liquidity and is not price-only. No IID p-values or profit claims are attached to overlapping labels.

In [ ]:
model = result['learning_report']
print('Validation-selected family:', model['selected_on_validation'])
print('Frozen date partitions:', model['split'])
display(pd.DataFrame([dict(model=name, test_rmse_bps=m['rmse_bps'], equal_date_mse_bps2=m['macro_day_mse_bps2'], dates=m['days'], negative_fraction=m['negative_prediction_fraction']) for name,m in model['test'].items()]))

## 3. Cont price-impact diagnostic
This contemporaneous fit explains midpoint changes using OFI over the same interval. It is **not** a forward forecast, paper replication or executable alpha.

In [ ]:
display(pd.DataFrame(result['impact']))

## 4. Separate numerical CVaR experiment
This section is always synthetic. A scalar constrained hedge is selected on exogenous scenarios and evaluated unchanged on new synthetic scenarios. The observed book/predictions above are deliberately not paired with unrelated simulated price paths as an allegedly calibrated joint distribution.

In [ ]:
hedge = synthetic_case()
print(hedge['assumptions'])
display(pd.DataFrame(hedge['held_out']).T)

## 5. Daily volatility models and optional browser report
HAR-RV, CHAR and CHAR_depth require a separately prepared complete daily panel with explicit calendar and variance conventions. Short capture intervals are not days. Use the CLI `har` command documented in the runbook.

To publish this analysis as a private interactive HTML/JSON report, explicitly set `DTS_ORDERBOOK_REPORT_DIR` to a new directory outside every Git checkout. Existing files are never overwritten. The report has no network requests and no live-trading capability.

In [ ]:
destination = os.environ.get('DTS_ORDERBOOK_REPORT_DIR')
if destination:
    print(write_report(Path(destination), result))
else:
    print('No report files requested; analysis is in memory only.')